# 17 · Cell registration (CellReg) — from the MATLAB run to a populated schema

**Purpose.** How longitudinal cell identity gets from a CellReg run on the server into
`cell_registration`, and how to query it afterwards. Ends with a worked example on one of
Leon's gratings sites.

> **Read this before running anything.**
> The `cell_registration` schema is **declared in code but not on the server**. The cells
> below that would *create* tables or *insert* rows are marked
> **`# REQUIRES AUTHORISATION`** and are deliberately left un-run. Creating the schema and
> populating it are production writes and need Tobias's explicit go-ahead in that session.
> Everything in §6 (querying) and §7 (the example) is read-only.

**Design review:** `adamacs/schemas/CELL_REGISTRATION_SCHEMA_DESIGN_REVIEW.md`
**Wiki:** `Database/Schemas/roselab_cell_registration.md`


---
## 1. Why this schema exists

Cross-session identity is currently one column: `imaging.Segmentation.Mask.roi_group`.
It is a `smallint`, it is **not part of the primary key**, and it is overwritten in place by
whichever CellReg ingest ran last. Nothing records which run produced it, with what
parameters, or what it held before.

That is wiki issue **D1** — counts recorded by an analysis (111, 146) disagree with the live
database (116, 120) — and **T1**: the delta cannot be reconstructed. It is also why
`adamacs_leon/helpers/pipeline_blocks.py` has `realign_global_ids_to_current_roi_group()`,
whose job is to repair identity after a re-ingest by *guessing* which CellReg column belongs
to which scan.

**What `roi_group` is, in CellReg terms:** the 0-based **row index of `cell_to_index_map`**.
This schema calls that `cell_index`. `roi_group` is now treated as a *derived projection* of
the latest curated run — a cache whose source is recorded, not an independent fact.

The one thing that makes an additive fix possible: a re-ingest rewrites `roi_group` but
**never the mask numbering**, so `(session_id, scan_id, paramset_idx, curation_id, mask)`
is a stable anchor.


---
## 2. Stage 1 — the MATLAB CellReg run on the server

Run on **Loki or ibehaveGPU, never tatchu3** (Tobias, 2025-08-26; a CellReg run exhausted
tatchu3's RAM in Feb 2025).

`Leon_CellReg.m` reads `converted_*.mat` (suite2p `Fall.mat` → CellReg footprints) and writes
into `<CR folder>/Results/`:

| File | Contents |
|---|---|
| `cellRegistered_<yyyymmdd_HHMMss>.mat` | `cell_registered_struct`: `cell_to_index_map`, the four score vectors, `p_same_registered_pairs`, `is_cell_in_overlapping_FOV`, centroids, alignment translations/rotations |
| `logFile_<yyyymmdd_HHMMss>.txt` | **the parameters and the run-level accuracy** |
| `Figures/` | CellReg's own diagnostics |

**Three facts that shape the importer** (measured over 108 files, 18.8 GB):

1. **The `.mat` contains no parameters at all.** They live only in `logFile_*.txt`, whose
   timestamp *trails* the result file (76 s in a verified pair), so pairing is "nearest log
   at or after the result file in the same directory".
2. **The score fields exist only for `registration_approach = 'Probabilistic'`**, and
   `alignment_rotations` only for `'Translations and Rotations'` (86 of 107 files).
3. **Footprints must stay on disk.** 93 of 107 files embed a stack; the largest is
   `513×513×3363` ≈ **7.1 GB** as float64. The importer never loads that field.


In [ ]:
import datetime
import os
import re

import datajoint as dj
import numpy as np

from adamacs.pipeline import imaging, scan, session, event    # activates the Elements
from adamacs.schemas import cell_registration as creg          # declares NOTHING on import
from adamacs.ingest import cellreg                             # the parsers

print("cell_registration activated on import:", creg.schema.is_activated())
print("parser version:", creg.PARSER_VERSION)

---
## 3. Inspect a run's files before touching the database

All of this is pure file reading — no connection needed.


In [ ]:
CR = ("/datajoint-data/data/leonk/analysis/cellreg/"
      "LE_ROS-1881_sess9FPXNI8U_Gratings_CR/Results/cellRegistered_20260129_075504.mat")

print("MAT format :", cellreg.detect_mat_format(CR))
print("fields     :", cellreg.list_fields(CR))
print("footprints :", cellreg.probe_footprints_storage(CR))   # metadata only; stack never read

log = cellreg.find_log_file(CR)
print("paired log :", log)

meta = cellreg.parse_log_file(open(log).read())
print("sessions   :", meta["n_sessions"], "| approach:", meta.get("registration_approach"))
print("alignment  :", meta.get("alignment_type"))
print("accuracy   : FP", meta.get("frac_false_positives"),
      "FN", meta.get("frac_false_negatives"),
      "uncertain", meta.get("frac_uncertain_pairs"))

# the log lists sessions in COLUMN order and the filenames carry the scan id
for i, f in enumerate(meta["session_files"]):
    print(f"   column {i}: {cellreg.scan_id_from_source_file(f)}")

---
## 4. Stage 2 — register the run  `# REQUIRES AUTHORISATION`

A run is identified by **the content of its result file**, not its path: 100 distinct
sha256 over 108 paths, because scan folders hold byte-identical backups. Keyed on path, one
registration would enter three times.

`CellRegRun.Session` carries `-> imaging.Curation`, so each row names the scan **and** the
suite2p paramset and curation it came from. A column that cannot be resolved to exactly one
curation goes to `CellRegRun.UnresolvedColumn` — it is never guessed.


In [ ]:
# REQUIRES AUTHORISATION — inserts rows. Left un-run deliberately.
#
# creg.activate(dj.config["custom"]["database.prefix"] + "cell_registration")
#
# sha = cellreg.file_sha256(CR)
# run_id = cellreg.run_id_from_sha(sha)
#
# creg.CellRegParamSet.register(
#     "leon_prob_sc_p03",
#     {"registration_approach": "probabilistic", "model_type": "spatial_correlation",
#      "p_same_threshold": meta["final_threshold"], "maximal_distance_um": meta["maximal_distance_um"],
#      "n_bins": meta["n_bins"], "initial_threshold": meta["initial_threshold"]},
#     label="Leon gratings/images standard",
#     matching_method="cellreg_probabilistic")
#
# creg.CellRegRun.insert1(dict(
#     cellreg_run_id=run_id, cellreg_paramset_id="leon_prob_sc_p03", subject="ROS-1881",
#     result_file=CR, result_file_sha256=sha, result_file_bytes=os.path.getsize(CR),
#     result_file_mtime=datetime.datetime.fromtimestamp(os.path.getmtime(CR)),
#     log_file=log, log_file_sha256=cellreg.file_sha256(log),
#     mat_format=cellreg.detect_mat_format(CR),
#     registration_approach=cellreg.normalise_registration_approach(meta["registration_approach"]),
#     alignment_mode=cellreg.normalise_alignment_mode(meta["alignment_type"]),
#     reference_session_index=meta.get("reference_session_index"),
#     n_sessions=meta["n_sessions"], n_registered_cells=meta["n_cells_final"],
#     ingested_at=datetime.datetime.now(), ingest_host=cellreg.ingest_host()))
#
# ... then one CellRegRun.Session row per resolved column, using
#     cellreg.resolve_column(column_masks, candidates, log_scan_id=...)
#     -> resolution_method, scan_key, mask_set_jaccard, ambiguous
print("not executed — see the banner at the top")

**Why Jaccard is computed even when the log resolves the column.** `Fall.mat` does **not**
update when suite2p curation does (Leon/Tobias, 2024-05-24), so a correctly-resolved column
can still carry a mask set that disagrees with the curation's. `mask_set_jaccard` is stored
unconditionally as a *staleness measurement*, not as a fallback.


---
## 5. Stage 3 — populate  `# REQUIRES AUTHORISATION`

`CellRegResult` is an `Imported` table: all `.mat` reading happens in its `make()`. It writes
per-cell quality as **rows with columns**, the cell↔session bridge, and `p_same` per cell per
session pair.

**Run it on a worker, not on tatchu3.** And watch the transaction: the largest run measures
~397,000 pair rows in a single `make()`.


In [ ]:
# REQUIRES AUTHORISATION — populates. Left un-run deliberately.
#
# creg.CellRegResult.populate(reserve_jobs=True, limit=50, max_calls=1, order="random",
#                             display_progress=True)
#
# A failed or inconsistent file is RECORDED, not skipped:
#   parse_ok = 0 with parse_error set, and no child rows.
#   p_same_consistent = 0 when the p_same matrix side != n_sessions, and no PairQuality rows.
print("not executed — see the banner at the top")

---
## 6. Stage 4 — querying

One edge attaches this schema to the pipeline: `CellRegRun.Session -> imaging.Curation`,
whose key is `(session_id, scan_id, paramset_idx, curation_id)`. `CellRegResult.Member`
inherits it and adds `mask`, so it lines up attribute-for-attribute with
`Segmentation.Mask` — joins need no renaming, and the curation can never be mixed implicitly.


In [ ]:
# Read-only patterns (will return nothing until the schema exists and is populated).
#
# run  = creg.CellRegRun & {"cellreg_run_id": run_id}
# cell = creg.CellRegResult.Cell & run & "cell_index = 42"
#
# # every (scan, mask) this cell is, across the sessions it was matched in
# members = creg.CellRegResult.Member & cell
#
# # the suite2p masks and traces those rows point at
# masks  = imaging.Segmentation.Mask   & members.proj()
# traces = imaging.Fluorescence.Trace  & members.proj() & "fluo_channel = 0"
#
# # cells tracked in >= 5 sessions whose weakest pairwise match is still good
# good = creg.CellRegResult.Cell & run & "n_sessions_present >= 5" & "min_p_same > 0.8"
#
# NOTE: n_sessions_present is a COUNT. Which sessions is a property of Member, not of Cell --
# Cell deliberately carries no session-scoped attribute. Partial tracking is the normal case:
# median sessions-per-cell is 1 in most runs.
print("query patterns above are illustrative")

---
## 7. Worked example — one gratings site, read-only

Site `sess9FPXNI8U` (ROS-1881), scan `scan9FPZGABI`, paramset 10, curation 2 — from the
`v1-drift-audit` `t78` validation set (11 sessions, 186 tracked cells,
`p_same_median` 0.98, `db_roi_group_equals_cellreg = 1.0`).

The figure below is built entirely from tables that exist **today**: the suite2p template
image from `imaging.MotionCorrection.Summary`, cell outlines from
`imaging.Segmentation.Mask`, and direction responses from `imaging.Activity.Trace`.


In [ ]:
SESS, SCAN, PIDX, CID = "sess9FPZGABI", "scan9FPZGABI", 10, 2
key = {"session_id": SESS, "scan_id": SCAN, "paramset_idx": PIDX, "curation_id": CID}

# suite2p template + masks that carry a roi_group
avg   = (imaging.MotionCorrection.Summary & key).fetch1("average_image")
masks = (imaging.Segmentation.Mask & key & "roi_group > 0").fetch(
    "mask", "roi_group", "mask_center_x", "mask_center_y", "mask_xpix", "mask_ypix",
    as_dict=True)
print("template:", avg.shape, "| masks with roi_group:", len(masks))

# Cascade inferred rate -- NOT suite2p_deconvolution, which is far too sparse to read
METHOD = "cascade_inference_25ms_filter"
print("extraction methods present:", sorted(set((imaging.Activity & key).fetch("extraction_method"))))

### The trap that costs an afternoon

**Cascade traces are NaN-padded at both ends** (wiki issue **T9**). A plain `.mean()` over a
trial window propagates the NaN, every response becomes NaN, every preferred orientation
becomes 0°, and the whole arm silently reports *no tuning* — for a non-biological reason.

Use NaN-aware means and require each window to be at least half finite. This notebook's
figure was empty on the first attempt for exactly this reason.


In [ ]:
from scipy import stats
from scipy.ndimage import gaussian_filter1d

# frame clock and grating trials
frame_t = np.sort((event.Event & {"session_id": SESS, "scan_id": SCAN,
                                  "event_type": "bench2p_frames"}).fetch("event_start_time").astype(float))
fps = 1.0 / np.median(np.diff(frame_t))
PRE, POST, STIM = int(round(1.0*fps)), int(round(4.0*fps)), int(round(3.0*fps))

def trial_matrix(trace, onsets):
    """Trials x time around onset, keeping only windows that are >= 50% finite (T9)."""
    rows = [trace[f-PRE:f+POST] for f in onsets if f-PRE >= 0 and f+POST <= trace.size]
    return np.vstack([r for r in rows
                      if np.isfinite(r[:PRE]).mean() >= 0.5
                      and np.isfinite(r[PRE:PRE+STIM]).mean() >= 0.5])

def direction_response(R):
    """Per-trial stimulus-minus-baseline, NaN-aware."""
    return np.nanmean(R[:, PRE:PRE+STIM], 1) - np.nanmean(R[:, :PRE], 1)

# cells were ranked by a one-way ANOVA of trial responses across the 12 directions,
# then thinned so no two chosen cells sit within 45 px of each other.
print(f"fps {fps:.2f} | window -1 s .. +4 s | stimulus {STIM} frames")

In [ ]:
# Build the figure. Selection: one-way ANOVA of trial responses across the 12 directions,
# then thin so no two chosen cells are within 45 px. Display smoothing 150 ms.
import matplotlib.pyplot as plt

dirs = sorted({int(re.match(r"Moving_Gratings:(\d+);", e).group(1))
               for e in set((event.Event & {"session_id": SESS, "scan_id": SCAN}).fetch("event_type"))
               if e.startswith("Moving_Gratings:")})
onset = {d: [int(np.searchsorted(frame_t, t0))
             for t0 in (event.Event & {"session_id": SESS, "scan_id": SCAN,
                                       "event_type": [e for e in
                                        set((event.Event & {"session_id": SESS, "scan_id": SCAN}
                                             ).fetch("event_type"))
                                        if e.startswith(f"Moving_Gratings:{d};")][0]}
                        ).fetch("event_start_time")] for d in dirs}

traces = {int(r["mask"]): np.asarray(r["activity_trace"], np.float32)
          for r in (imaging.Activity.Trace & key & {"extraction_method": METHOD}
                    ).fetch("mask", "activity_trace", as_dict=True)}
mask_by_id = {int(m["mask"]): m for m in masks}

seg, score, prefd = {}, {}, {}
for m in sorted(set(mask_by_id) & set(traces)):
    tr = traces[m]
    if np.isfinite(tr).sum() < 0.5 * tr.size:          # T9
        continue
    per_dir, groups, ok = {}, [], True
    for d in dirs:
        R = trial_matrix(tr, onset[d])
        if R.shape[0] < 10:
            ok = False; break
        per_dir[d] = R
        g = direction_response(R)
        if not np.isfinite(g).all():
            ok = False; break
        groups.append(g)
    if not ok:
        continue
    F, _ = stats.f_oneway(*groups)
    mu = np.array([g.mean() for g in groups])
    if not np.isfinite(F) or mu.max() <= 0:
        continue
    seg[m], score[m], prefd[m] = per_dir, float(F), dirs[int(np.argmax(mu))]

chosen, cents = [], []
for m in sorted(score, key=score.get, reverse=True):
    cxm, cym = mask_by_id[m]["mask_center_x"], mask_by_id[m]["mask_center_y"]
    if all((cxm - a) ** 2 + (cym - b) ** 2 > 45 ** 2 for a, b in cents):
        chosen.append(m); cents.append((cxm, cym))
    if len(chosen) == 7:
        break
print("chosen:", [(m, round(score[m], 1), prefd[m]) for m in chosen])

CROP, UMPX, SM = 30, 615.6 / 512, 0.15 * fps
FOCAL, OTHER, BAND = "#12355b", "#6e8cae", "#b9cde2"
fig = plt.figure(figsize=(7.2, 2.62))
gs = fig.add_gridspec(2, len(chosen), height_ratios=[1.0, 0.86], hspace=0.09, wspace=0.04,
                      left=0.010, right=0.990, top=0.878, bottom=0.135)
for i, m in enumerate(chosen):
    md_ = mask_by_id[m]
    ax = fig.add_subplot(gs[0, i])
    x0 = int(np.clip(md_["mask_center_x"] - CROP, 0, avg.shape[1] - 2 * CROP))
    y0 = int(np.clip(md_["mask_center_y"] - CROP, 0, avg.shape[0] - 2 * CROP))
    sub = avg[y0:y0 + 2 * CROP, x0:x0 + 2 * CROP]
    p2, p99 = np.percentile(sub, [2, 99.5])
    ax.imshow(sub, cmap="gray", vmin=p2, vmax=p99, interpolation="bilinear")
    mm = np.zeros(avg.shape, bool); mm[md_["mask_ypix"], md_["mask_xpix"]] = True
    ax.contour(mm[y0:y0 + 2 * CROP, x0:x0 + 2 * CROP], levels=[0.5], colors=FOCAL, linewidths=1.1)
    ax.set_axis_off()
    ax.text(0.5, 1.03, f"cell {md_['roi_group']}", transform=ax.transAxes,
            ha="center", va="bottom", fontsize=6.2, color="0.15")
    if i == 0:
        px = 25 / UMPX
        ax.plot([3.5, 3.5 + px], [2 * CROP - 4.5] * 2, lw=1.8, color="w", solid_capstyle="butt")
        ax.text(3.5 + px / 2, 2 * CROP - 6.5, "25 µm", ha="center", va="bottom",
                fontsize=5.5, color="w")

    ax = fig.add_subplot(gs[1, i])
    curves = []
    for d in dirs:
        R = seg[m][d].astype(np.float64)
        R = np.where(np.isfinite(R), R, np.nanmean(R))
        R = gaussian_filter1d(R, SM, axis=1, mode="nearest")
        n = R.shape[0]
        curves.append((R.mean(0), stats.t.ppf(0.975, n - 1) * R.std(0, ddof=1) / np.sqrt(n)))
    peak = max(float((mu_ + hw_).max()) for mu_, hw_ in curves)
    nT = curves[0][0].size
    ipref = dirs.index(prefd[m])
    for dd, (mu_, hw_) in enumerate(curves):
        xs = np.arange(nT) + dd * nT
        ax.fill_between(xs, (mu_ - hw_) / peak, (mu_ + hw_) / peak, color=BAND, lw=0,
                        alpha=0.9, zorder=1)
        ax.plot(xs, mu_ / peak, color=FOCAL if dd == ipref else OTHER,
                lw=1.0 if dd == ipref else 0.6, zorder=3 if dd == ipref else 2)
        ax.plot([dd * nT + PRE, dd * nT + PRE + STIM], [-0.095] * 2, lw=1.0, color="0.78",
                solid_capstyle="butt", zorder=1)
    ax.set_xlim(-3, len(dirs) * nT + 3); ax.set_ylim(-0.30, 1.08); ax.set_axis_off()
    ax.text(ipref * nT + nT / 2, 1.07, f"{prefd[m]}°", ha="center", va="top",
            fontsize=5.8, color=FOCAL)
    if i == 0:
        ax.plot([0, fps], [-0.20] * 2, lw=1.5, color="0.15", solid_capstyle="butt")
        ax.text(fps * 0.5, -0.225, "1 s", ha="center", va="top", fontsize=5.5, color="0.15")

fig.text(0.5, 0.997, "Longitudinally registered cells and their direction tuning  ·  "
         f"{SCAN}  ·  paramset {PIDX}, curation {CID}", ha="center", va="top",
         fontsize=7.5, color="0.10")
fig.text(0.5, 0.006, "per cell: 12 directions 0–330° left to right, mean ± 95% CI over 15 trials  ·  "
         "dark trace and label = preferred direction  ·  grey rule = 3 s drifting grating  ·  "
         "Cascade inferred rate (25 ms filter), 150 ms smoothing, each cell scaled to its own peak",
         ha="center", va="bottom", fontsize=5.5, color="0.36")
fig.savefig("figures/fig_cellreg_tuning.png", dpi=400)   # gitignored; regenerate by running


### Result

*(`notebooks/figures/*.png` is gitignored, so the figure is not committed — running the cell above regenerates it.)*

![Longitudinally registered cells and their direction tuning]({{artifact:bbe4781f-ca72-463e-b398-bf968993df08}})

Seven direction-tuned cells from one scan (one-way ANOVA across the 12 directions,
F = 32.3–48.5). Top: each cell outlined on the suite2p average image, contrast set per crop.
Bottom: the 12 directions laid end to end, mean ± 95% CI over 15 trials, dark trace marking
the preferred direction. Preferred directions span 0–240°, which is the point — these are
*different* cells, and the registration is what lets the same seven be followed across the
site's other sessions.


---
## 8. Gotchas, in the order they will bite you

| | |
|---|---|
| **Cascade NaN padding (T9)** | NaN-pad at both ends; a plain mean silently yields zero tuning. Use NaN-aware means, require ≥50 % finite. |
| **`suite2p_deconvolution` is sparse** | Fine for statistics, unreadable as a plotted trace. Use a Cascade variant for figures. |
| **Table-name prefixes** | `MotionCorrection` is `Imported` → `_motion_correction__summary`. `Segmentation` is `Computed` → `__segmentation__mask`. |
| **Blobs need DataJoint** | Fetch through virtual modules or the activated pipeline. Raw SQL + `datajoint.blob.unpack` fails on externally-stored blobs. |
| **`event.Event` is ~1.05 billion rows** | Always restrict by `session_id` *and* `scan_id` first (wiki **D5**). An unrestricted count will hang. |
| **Natural-image vs gratings scans** | Same site, different scans, **separate CellReg runs**. `scan9FPZHKKB` is images; `scan9FPZGABI` is gratings. |
| **Two runs in one `Results/`** | ROS-1985 has a 7-session run and a 14-session run side by side (wiki **D3**). Pair the log to the result file; do not assume one run per folder. |
| **`same_site_id` cannot say "unknown"** | Populated for all 3,460 sessions but 74 % self-reference. 41 sites carry real groupings (up to 66 sessions). Treat a self-reference as unknown, not as a singleton site. |
| **Not on tatchu3** | CellReg and bulk `.mat` parsing go to Loki or ibehaveGPU. |


---
## 9. What is deliberately *not* here

- **Eligibility thresholds.** Registration stores continuous match quality only; deciding
  which cells are good enough is a QC/cohort act with its own version history.
- **`MaskCoordinates.make()`** raises `NotImplementedError` pending verification that
  `centroid_locations_corrected` is ordered consistently with `Segmentation.Mask` — declaring
  it without that check would invite wiki issue **L8**.
- **Writing `roi_group`.** `RoiGroupProjection` records what a projection *would* be;
  performing one is a production write needing explicit authorisation.
